In [9]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq

In [2]:
def load_document(file_path):
    loader = TextLoader(file_path)
    return loader.load()[0].page_content

In [3]:
def chunk_document(raw_text, file_name):
    splitter = MarkdownHeaderTextSplitter([
        ("#", "Doc"),
        ("##", "Section")
    ])
    chunks = splitter.split_text(raw_text)
    for c in chunks:
        c.metadata["file"] = file_name
    return chunks

In [4]:
files = [
    "01_mgc_aurora_heights_brochure.md",
    "02_price_list_payment_plan.md",
    "03_booking_policy_faq.md"
]


all_chunks = []
for f in files:
    raw = load_document(f)
    chunks = chunk_document(raw, f)
    all_chunks.extend(chunks)

print(f" Total chunks: {len(all_chunks)}")

 Total chunks: 22


In [5]:
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5")

vectorstore = Chroma.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    persist_directory="./chroma_db",      
    collection_name="mgc_docs"            
)
vectorstore.persist()

print(" Vectorstore saved to ./chroma_db")

C:\Users\hp\AppData\Local\Temp\ipykernel_20960\1129263250.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5")
d:\MGC\mgc-task-pack\mcg_task\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\MGC\mgc-task-pack\mcg_task\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hp\.cac

 Vectorstore saved to ./chroma_db


C:\Users\hp\AppData\Local\Temp\ipykernel_20960\1129263250.py:9: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


In [7]:
load_dotenv()
def load_key(key_name):
    try:
        return os.environ[key_name]
    except KeyError:
        raise KeyError(f"Environment variable '{key_name}' not found. Please set it in your .env file.")

key=load_key("groq")
     


In [10]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    api_key = key,
    temperature = 0,
    max_tokens = None,
    timeout=None,
    max_retries = 2
)

In [25]:
def ask(question):
    # Retrieve
    docs = retriever.invoke(question)
    
    # Build context with sources
    ctx = "\n\n".join(
        f"{d.page_content}\n[Source: {d.metadata.get('file', 'unknown')} | {d.metadata.get('Section', 'General')}]"
        for d in docs
    )
    
    # Prompt with rules
    prompt = f"""Answer ONLY from the context. If not found, say "I don't have that, ask the marketing manager."
If conflicting info appears, flag it and list both sources. Never invent.

Context:
{ctx}

Question: {question}

Answer:"""
    
    # Generate
    answer = llm.invoke(prompt).content
    return answer, docs

In [33]:
question = "Base price of a 2-bed in Block B standard?"
answer, sources = ask(question)

In [34]:
print(answer)

22,425,000
